###### Developed by : Karthika
###### Created Date: 18-July- 2026
###### updates:
        - Fixed the latitude and longitude error
        - fixed the trust warning becasue folium map was not loading.
        - 22-July-2026: switched keyword matching from plain substring to word-boundary
          matching (avoids false matches, e.g. "AI" inside "training").
        - 22-July-2026: added profile_fit_score (from the ProfileEncoder) as a weighted
          input to job scoring.
        - 22-July-2026: added plain-language explanations for each job + housing
          recommendation, for use in the Streamlit UI.


In [1]:
# ---------------------------------------------------------------------
# 1. Import Libraries
# ---------------------------------------------------------------------

import re
from math import radians, sin, cos, sqrt, atan2

import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# ---------------------------------------------------------------------
# 2. Newcomer's profile 
# ---------------------------------------------------------------------

user_profile = {
    "occupation_category": "Business, Finance and Administration",
    "previous_occupation": "Data Analyst",
    "field_of_study": "Computer Science",
    "years_of_experience": 6,
    "TEER_category": "TEER 1",
    "regulated_profession": False,
    "credential_recognition_status": "Fully Recognized",
    "additional_skills": ["SQL", "Power BI", "SSIS"], 
    "preferred_cities": ["Toronto", "Mississauga", "Burlington", "Hamilton", "Oakville"],
    "minimum_salary": 65000,
    "bedroom_type": "2 Bedroom",
    "max_commute_km": 30,
    "max_rent_income_ratio": 0.30,
    "top_jobs": 10,
    "homes_per_job": 5,
    "employment_probability": 0.75,   # placeholder -- swap for employment_model.predict_proba(...)
    "predicted_income": 68000,    # placeholder -- swap for income_model.predict(...)
    "profile_fit_score": 0.86,    # placeholder -- swap for ProfileEncoder embedding-based fit score
}

display(pd.DataFrame([user_profile]))


,occupation_category,previous_occupation,field_of_study,years_of_experience,TEER_category,regulated_profession,credential_recognition_status,additional_skills,preferred_cities,minimum_salary,bedroom_type,max_commute_km,max_rent_income_ratio,top_jobs,homes_per_job,employment_probability,predicted_income,profile_fit_score
0,"Business, Finance and Administration",Data Analyst,Computer Science,6,TEER 1,False,Fully Recognized,"[SQL, Power BI, SSIS]","[Toronto, Mississauga, Burlington, Hamilton, O...",65000,2 Bedroom,30,0.3,10,5,0.75,68000,0.86


In [3]:
# ---------------------------------------------------------------------
# 3. Load the job postings data (Adzuna)
# ---------------------------------------------------------------------

job_df = pd.read_csv("processed_adzuna_jobs.csv")

# Rename columns to simple names, and combine salary_min/salary_max into one salary column
job_df = job_df.rename(columns={"latitude": "lat", "longitude": "lon"})
job_df["salary"] = job_df[["salary_min", "salary_max"]].mean(axis=1)

# Drop postings with no coordinates because some remote jobs have none
job_df = job_df.dropna(subset=["lat", "lon"])

print(f"Jobs loaded: {len(job_df)}")
display(job_df[["title", "company", "location", "salary", "category"]].head())

Jobs loaded: 358


,title,company,location,salary,category
0,Data Analyst,Insight Global,"Toronto, Ontario",NaN,IT Jobs
1,Data Analyst,ADF Medical Services Inc.,"Toronto Dominion Centre, City of Toronto",NaN,IT Jobs
2,Data Analyst,Insight Global,"Toronto, Ontario",NaN,IT Jobs
3,Data Analyst,Delpath,"Toronto, Ontario",NaN,IT Jobs
4,Data Analyst,Vretta,"Toronto, Ontario",NaN,IT Jobs


In [4]:
# ---------------------------------------------------------------------
# 4. Analyze the job data
# ---------------------------------------------------------------------
print("Column info:")
print(job_df.info())

print("\nHow many postings have a salary listed?")
print(job_df["salary"].notna().value_counts())

print("\nTop job categories:")
print(job_df["category"].value_counts().head())

Column info:
<class 'pandas.core.frame.DataFrame'>
Index: 358 entries, 0 to 368
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   358 non-null    int64  
 1   title                358 non-null    object 
 2   company              358 non-null    object 
 3   location             358 non-null    object 
 4   salary_min           71 non-null     float64
 5   salary_max           69 non-null     float64
 6   salary_is_predicted  358 non-null    int64  
 7   lat                  358 non-null    float64
 8   lon                  358 non-null    float64
 9   description          358 non-null    object 
 10  category             358 non-null    object 
 11  category_tag         358 non-null    object 
 12  contract_type        67 non-null     object 
 13  contract_time        147 non-null    object 
 14  created              358 non-null    object 
 15  url                  358 non-nul

In [5]:
#----------------------------------------------------------------------
# 5. Load the Housing Data (CMHC)
#----------------------------------------------------------------------
housing_df = pd.read_csv("housing_geocoded.csv")

housing_df = housing_df.rename(columns={
    "latitude": "lat",
    "longitude": "lon",
    "rent_price": "monthly_rent",
})

print(f"Housing records loaded: {len(housing_df)}")
display(housing_df[["city", "neighbourhood", "bedroom_type", "monthly_rent", "lat", "lon"]].head())

Housing records loaded: 2000


,city,neighbourhood,bedroom_type,monthly_rent,lat,lon
0,Barrie,Barrie,2 Bedroom,1755,44.3894,-79.6903
1,Belleville,Belleville - Quinte West,2 Bedroom,1484,44.1628,-77.3832
2,Brantford,Brantford,2 Bedroom,1619,43.1394,-80.2644
3,Guelph,Guelph,2 Bedroom,1802,43.5448,-80.2482
4,Hamilton,Hamilton,2 Bedroom,1656,43.2557,-79.8711


In [6]:
#----------------------------------------------------------------------
# 6. Explore the Housing Data
#----------------------------------------------------------------------
print("Rent summary statistics:")
print(housing_df["monthly_rent"].describe())

print("\nNumber of listings per city:")
print(housing_df["city"].value_counts())

Rent summary statistics:
count    2000.00000
mean     1572.61800
std       336.67536
min       830.00000
25%      1328.75000
50%      1535.00000
75%      1770.00000
max      2900.00000
Name: monthly_rent, dtype: float64

Number of listings per city:
city
Guelph                144
Toronto               136
Ottawa                135
Brantford             134
Belleville            132
Windsor               128
Peterborough          127
Oshawa                126
St. Catharines        125
Greater Sudbury       125
Kitchener-Waterloo    122
Thunder Bay           121
Barrie                119
Kingston              112
London                110
Hamilton              104
Name: count, dtype: int64


In [7]:
# ---------------------------------------------------------------------
# 7. Text cleaning and keyword matching function
# ---------------------------------------------------------------------

def clean_text(text):
    # converted the text to lowercase and removed the punctuation
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return text


def count_matching_keywords(text, keyword_list):
    # Checks how many keywords from keyword_list appear inside text as whole words,
    # not as substrings (so "AI" won't match inside "training" or "detail").
    # Returns a score between 0 and 1

    if len(keyword_list) == 0:
        return 0.0

    cleaned_text = clean_text(text)
    match_count = 0

    for keyword in keyword_list:
        cleaned_keyword = clean_text(keyword).strip()
        if not cleaned_keyword:
            continue
        pattern = r"\b" + re.escape(cleaned_keyword) + r"\b"
        if re.search(pattern, cleaned_text):
            match_count = match_count + 1

    return match_count / len(keyword_list)


def city_is_preferred(location, preferred_cities):
    # Returns 1 if the job and housing location matches one of the preferred cities, else 0.
    cleaned_location = clean_text(location)
    for city in preferred_cities:
        if clean_text(city).strip() in cleaned_location:
            return 1.0
    return 0.0


# unit test to see if the above functions is working
test_score = count_matching_keywords("Data Analyst with SQL and Power BI experience", user_profile["additional_skills"])
print(f"Test: how well does a sample job description match our skills list? Score = {test_score:.2f}")

# regression test: word-boundary fix should NOT match "AI" inside "training"
false_match_check = count_matching_keywords("Warehouse training coordinator", ["AI"])
print(f"Test: 'AI' should NOT match inside 'training' -> Score = {false_match_check:.2f} (expected 0.00)")


Test: how well does a sample job description match our skills list? Score = 0.67
Test: 'AI' should NOT match inside 'training' -> Score = 0.00 (expected 0.00)


In [8]:
# ---------------------------------------------------------------------
# 8. salary scoring function
# ---------------------------------------------------------------------

def score_salary(salary, minimum_salary):
    # generates a score between 0 and 1 based on how the salary compares to the minimum wanted.

    if pd.isna(salary) or salary <= 0:
        return 0.35  # unknown salary gets a neutral, low-ish score
    if salary >= minimum_salary:
        return 1.0
    return salary / minimum_salary

# unit test
print("Salary $80,000 vs minimum $65,000 ->", score_salary(80000, 65000))
print("Salary $50,000 vs minimum $65,000 ->", score_salary(50000, 65000))
print("No salary listed ->", score_salary(None, 65000))

Salary $80,000 vs minimum $65,000 -> 1.0
Salary $50,000 vs minimum $65,000 -> 0.7692307692307693
No salary listed -> 0.35


In [9]:
# ---------------------------------------------------------------------
# 9. score and rank all jobs against the profile
# ---------------------------------------------------------------------

def rank_jobs(jobs_df, profile):
    # Scores every job against the user's profile and returns the top-ranked jobs
    jobs = jobs_df.copy()
    scores = []

    # Combine the profile fields into one list of terms to match
    profile_terms = [
        profile.get("occupation_category", ""),
        profile.get("previous_occupation", ""),
        profile.get("field_of_study", ""),
    ] + profile.get("additional_skills", [])
    profile_terms = [t for t in profile_terms if t]  # dropping empty values

    # Employment Model connection: if not provided, assume neutral
    employment_factor = profile.get("employment_probability", 1.0)

    # ProfileEncoder connection: profile fit score (0-1). If not provided, assume neutral (0.5)
    profile_fit_score = profile.get("profile_fit_score", 0.5)

    for index, job in jobs.iterrows():
        job_text = str(job["title"]) + " " + str(job["description"]) + " " + str(job["category"])

        # Here we are checking how well does the job is a match to the newcomer's occupational background
        background_score = count_matching_keywords(job_text, profile_terms)

        # Title match specifically against their previous occupation
        title_score = count_matching_keywords(str(job["title"]), [profile.get("previous_occupation", "")])

        salary_score = score_salary(job["salary"], profile["minimum_salary"])
        city_score = city_is_preferred(str(job["location"]), profile["preferred_cities"])

        # Combine the five scores into one overall score out of 100, weighted:
        # background 35%, title 20%, salary 20%, city 10%, ProfileEncoder fit 15%
        # then scale by employment probability
        content_score = (
            0.35 * background_score
            + 0.20 * title_score
            + 0.20 * salary_score
            + 0.10 * city_score
            + 0.15 * profile_fit_score
        )
        total_score = 100 * content_score * employment_factor
        scores.append(total_score)

    jobs["job_score"] = scores
    jobs = jobs.sort_values("job_score", ascending=False)
    return jobs.head(profile["top_jobs"]).reset_index(drop=True)

top_jobs = rank_jobs(job_df, user_profile)
print(f"Employment probability used in scoring: {user_profile['employment_probability']:.0%}")
print(f"ProfileEncoder fit score used in scoring: {user_profile['profile_fit_score']:.0%}")
print(f"Top {len(top_jobs)} jobs for this profile:")
display(top_jobs[["title", "company", "location", "salary", "job_score"]].round({"salary": 0, "job_score": 1}))


Employment probability used in scoring: 75%
ProfileEncoder fit score used in scoring: 86%
Top 10 jobs for this profile:


,title,company,location,salary,job_score
0,"Senior Business Data Analyst, Performance Mana...",LifeLabs,"Toronto, Ontario",92628.0,55.9
1,Data Analyst,CI Financial,"Toronto, Ontario",75000.0,51.6
2,Data Analyst,Acquird.io,"Toronto, Ontario",75000.0,51.6
3,Senior Data Analyst,Konrad,"Toronto, Ontario",110000.0,51.6
4,Senior Data Analyst (m/f/d),AutoTrader.ca,"Toronto, Ontario",125000.0,51.6
5,Senior Financial Data Analyst,SimCorp,"Toronto, Ontario",111200.0,51.6
6,Data Analyst,Konrad,"Toronto, Ontario",87500.0,51.6
7,Data Analyst,CI Financial,"Toronto, Ontario",75000.0,51.6
8,"Senior AI Analytics, Data Analyst",Parent Organization,"Regent Park, City of Toronto",159120.0,51.6
9,Lead Healthcare Research & Data Analyst,Clarivate,"Toronto, Ontario",99000.0,51.6


In [10]:
# ---------------------------------------------------------------------
# 10. Distance calculation (Haversine Formula)
# ---------------------------------------------------------------------

def haversine_distance(lat1, lon1, lat2, lon2):
    # Calculates the distance in kilometers between two geotagged points,latitude/longitude coordinates

    earth_radius_km = 6371
    lat1, lon1, lat2, lon2 = radians(lat1), radians(lon1), radians(lat2), radians(lon2)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = sin(delta_lat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(delta_lon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return earth_radius_km * c

# unit test
test_distance = haversine_distance(43.6532, -79.3832, 43.2557, -79.8711)
print(f"Test: distance from Toronto to Hamilton = {test_distance:.1f} km")

Test: distance from Toronto to Hamilton = 59.2 km


In [11]:
from housing_recommendation_module import recommend_housing, OSRMDistanceProvider

osrm_provider = OSRMDistanceProvider(cache_path="osrm_cache.json")

recommendations_osrm = recommend_housing(
    profile=user_profile,
    ranked_jobs=top_jobs,
    housing=housing_df,
    distance_provider=osrm_provider,
)

recommendations_osrm.head()

,record_id,city,cmhc_centre,neighbourhood,bedroom_type,unit_type,monthly_rent,cmhc_avg_rent_2br_2025,vacancy_rate_2025,turnover_rate_2025,...,commute_category,score_breakdown,affordability_score,commute_score,bedroom_score,city_score,housing_score,housing_match_level,combined_score,explanation
0,SYN-1353,Toronto,Toronto CMA,Toronto area 03,2 Bedroom,Apartment,2205.0,2046,3.0,8.7,...,Short,"{'affordability': 0.984, 'commute': 0.9467, 'b...",0.9840,0.9467,1.0,1.0,97.68,Excellent Match,72.31,"[The monthly rent of $2,205 is above the estim..."
1,SYN-0179,Toronto,Toronto CMA,Toronto area 09,2 Bedroom,Townhouse,2420.0,2046,3.0,8.7,...,Short,"{'affordability': 0.9099, 'commute': 0.8042, '...",0.9099,0.8042,1.0,1.0,90.07,Excellent Match,71.29,"[The monthly rent of $2,420 is above the estim..."
2,SYN-1097,Toronto,Toronto CMA,Toronto area 12,2 Bedroom,Apartment,2195.0,2046,3.0,8.7,...,Short,"{'affordability': 0.9931, 'commute': 0.8297, '...",0.9931,0.8297,1.0,1.0,94.58,Excellent Match,70.91,"[The monthly rent of $2,195 is above the estim..."
3,SYN-0776,Toronto,Toronto CMA,Toronto area 06,2 Bedroom,Townhouse,2355.0,2046,3.0,8.7,...,Short,"{'affordability': 0.9661, 'commute': 0.6176, '...",0.9661,0.6176,1.0,1.0,87.00,Excellent Match,69.91,"[The monthly rent of $2,355 is above the estim..."
4,SYN-1288,Toronto,Toronto CMA,Toronto area 07,2 Bedroom,Townhouse,2375.0,2046,3.0,8.7,...,Short,"{'affordability': 0.9488, 'commute': 0.6217, '...",0.9488,0.6217,1.0,1.0,86.35,Excellent Match,69.61,"[The monthly rent of $2,375 is above the estim..."


In [12]:
# ---------------------------------------------------------------------
# 11. score housing options for a chosen job
# ---------------------------------------------------------------------

def max_affordable_rent(annual_salary, profile):
    # Calculates the highest monthly rent this salary can afford, using the 30% rule.
    # Here we will connect the Income model to housing recommendations.

    if pd.isna(annual_salary) or annual_salary <= 0:
        annual_salary = profile.get("predicted_income", profile["minimum_salary"])
    monthly_salary = annual_salary / 12
    return monthly_salary * profile["max_rent_income_ratio"]


def rank_housing_for_job(housing_df, job, profile):
    # For one specific job, scores every housing listing based on affordability, commute distance, bedroom type, and city preference.

    housing = housing_df.copy()

    salary = job["salary"]
    affordable_rent = max_affordable_rent(salary, profile)

    commute_list = []
    affordability_score_list = []
    bedroom_score_list = []
    city_score_list = []

    for index, home in housing.iterrows():
        # Commute distance
        distance = haversine_distance(home["lat"], home["lon"], job["lat"], job["lon"])
        commute_list.append(distance)

        # Affordability score: closer or under budget = better score
        if home["monthly_rent"] <= affordable_rent:
            score = 1 - (home["monthly_rent"] / affordable_rent)
            score = max(0, min(1, score))
        else:
            score = (affordable_rent / home["monthly_rent"]) * 0.5
        affordability_score_list.append(score)

        # Bedroom match
        if clean_text(home["bedroom_type"]).strip() == clean_text(profile["bedroom_type"]).strip():
            bedroom_score_list.append(1.0)
        else:
            bedroom_score_list.append(0.0)

        # City preference match
        city_score_list.append(city_is_preferred(str(home["city"]), profile["preferred_cities"]))

    housing["commute_km"] = commute_list
    housing["affordability_score"] = affordability_score_list
    housing["bedroom_score"] = bedroom_score_list
    housing["housing_city_score"] = city_score_list
    housing["max_affordable_rent"] = affordable_rent
    housing["affordable"] = housing["monthly_rent"] <= affordable_rent

    # Commute score: closer commute = higher score
    commute_scores = []
    for distance in housing["commute_km"]:
        score = 1 - (distance / profile["max_commute_km"])
        score = max(0, min(1, score))
        commute_scores.append(score)
    housing["commute_score"] = commute_scores

    housing["housing_score"] = 100 * (
        0.45 * housing["affordability_score"]
        + 0.30 * housing["commute_score"]
        + 0.15 * housing["bedroom_score"]
        + 0.10 * housing["housing_city_score"]
    )

    housing["job_title"] = job["title"]
    housing["job_company"] = job["company"]
    housing["job_salary"] = salary
    housing["job_score"] = job["job_score"]

    # Combined score: 55% job fit, 45% housing fit
    housing["combined_score"] = 0.55 * housing["job_score"] + 0.45 * housing["housing_score"]

    housing = housing.sort_values("combined_score", ascending=False)
    return housing.head(profile["homes_per_job"]).reset_index(drop=True)

# unit test with just the #1 top job to check the output
best_job = top_jobs.iloc[0]
print(f"Testing housing scoring for the top job: {best_job['title']} at {best_job['company']}")
best_job_housing = rank_housing_for_job(housing_df, best_job, user_profile)
display(best_job_housing[["city", "neighbourhood", "monthly_rent", "commute_km", "affordable", "housing_score", "combined_score"]].round(1))

Testing housing scoring for the top job: Senior Business Data Analyst, Performance Management at LifeLabs


,city,neighbourhood,monthly_rent,commute_km,affordable,housing_score,combined_score
0,Toronto,Toronto area 09,2420,5.3,False,71.2,62.8
1,Toronto,Toronto area 06,2355,6.9,False,70.3,62.4
2,Toronto,Toronto area 08,2320,13.4,False,64.0,59.6
3,Toronto,Toronto area 07,2375,13.5,False,63.4,59.3
4,Toronto,Toronto area 12,1845,2.9,True,61.2,58.3


In [13]:
# ---------------------------------------------------------------------
# 12b. Plain-language explanation for each job + housing recommendation
# ---------------------------------------------------------------------

def generate_recommendation_explanation(row, profile):
    # Builds a short list of plain-language reasons why this job + housing
    # combination was recommended, for display in the Streamlit app.
    reasons = []

    if row.get("job_score", 0) >= 60:
        reasons.append("This job scored highly against your occupation background, skills, and salary expectations.")
    elif row.get("job_score", 0) >= 40:
        reasons.append("This job is a reasonable match for your background and salary expectations.")

    if profile.get("profile_fit_score", 0) >= 0.70:
        reasons.append("Your profile embedding shows a strong overall fit with this type of role.")

    if row.get("affordable", False):
        reasons.append(f"The rent (${row['monthly_rent']:.0f}/month) fits within 30% of your expected income.")
    else:
        reasons.append(f"Note: the rent (${row['monthly_rent']:.0f}/month) is above the affordable threshold for this job's salary.")

    if row.get("commute_km", 999) <= profile.get("max_commute_km", 30):
        reasons.append(f"The commute is about {row['commute_km']:.1f} km, within your preferred range.")
    else:
        reasons.append(f"The commute is about {row['commute_km']:.1f} km, longer than your preferred range.")

    if row.get("housing_city_score", 0) == 1.0:
        reasons.append(f"This housing option is in one of your preferred cities ({row['city']}).")

    return " ".join(reasons)


# unit test using the top housing match for the top job
sample_row = best_job_housing.iloc[0]
print(generate_recommendation_explanation(sample_row, user_profile))


This job is a reasonable match for your background and salary expectations. Your profile embedding shows a strong overall fit with this type of role. Note: the rent ($2420/month) is above the affordable threshold for this job's salary. The commute is about 5.3 km, within your preferred range. This housing option is in one of your preferred cities (Toronto).


In [14]:
# ---------------------------------------------------------------------
# 13. Run housing scoring for all top jobs and combine the results
# ---------------------------------------------------------------------

def build_recommendations(job_df, housing_df, profile):
    # Runs the full pipeline: rank jobs, then rank housing for each top job.
    top_jobs = rank_jobs(job_df, profile)

    all_recommendations = []
    for index, job in top_jobs.iterrows():
        housing_matches = rank_housing_for_job(housing_df, job, profile)
        all_recommendations.append(housing_matches)

    combined = pd.concat(all_recommendations, ignore_index=True)
    combined = combined.sort_values("combined_score", ascending=False)
    combined = combined.reset_index(drop=True)

    # Add a plain-language explanation for each row, for display in Streamlit
    combined["explanation"] = combined.apply(
        lambda row: generate_recommendation_explanation(row, profile), axis=1
    )
    return combined

recommendations = build_recommendations(job_df, housing_df, user_profile)
print(f"Total job + housing combinations scored: {len(recommendations)}")
print("\nTop 10 overall recommendations:")
display(
    recommendations[[
        "job_title", "job_company", "city", "neighbourhood", "monthly_rent",
        "commute_km", "affordable", "combined_score"
    ]].head(10).round({"monthly_rent": 0, "commute_km": 1, "combined_score": 1})
)

print("\nExample explanation for the top recommendation:")
print(recommendations.iloc[0]["explanation"])


Total job + housing combinations scored: 50

Top 10 overall recommendations:


,job_title,job_company,city,neighbourhood,monthly_rent,commute_km,affordable,combined_score
0,"Senior Business Data Analyst, Performance Mana...",LifeLabs,Toronto,Toronto area 09,2420,5.3,False,62.8
1,Data Analyst,Konrad,Toronto,Toronto area 03,2205,1.2,False,62.6
2,"Senior Business Data Analyst, Performance Mana...",LifeLabs,Toronto,Toronto area 06,2355,6.9,False,62.4
3,"Senior AI Analytics, Data Analyst",Parent Organization,Toronto,Toronto,2046,1.6,True,62.2
4,"Senior AI Analytics, Data Analyst",Parent Organization,Toronto,Toronto area 01,1945,3.2,True,62.0
5,Data Analyst,Konrad,Toronto,Toronto area 12,2195,3.8,False,61.5
6,Data Analyst,CI Financial,Toronto,Toronto area 03,2205,1.2,False,61.2
7,Data Analyst,CI Financial,Toronto,Toronto area 03,2205,1.2,False,61.2
8,Data Analyst,Acquird.io,Toronto,Toronto area 03,2205,1.2,False,61.2
9,"Senior AI Analytics, Data Analyst",Parent Organization,Toronto,Toronto area 04,2065,4.2,True,61.0



Example explanation for the top recommendation:
This job is a reasonable match for your background and salary expectations. Your profile embedding shows a strong overall fit with this type of role. Note: the rent ($2420/month) is above the affordable threshold for this job's salary. The commute is about 5.3 km, within your preferred range. This housing option is in one of your preferred cities (Toronto).


In [15]:
# ---------------------------------------------------------------------
# 15. Regional Rent summary table
# ---------------------------------------------------------------------

summary = housing_df.groupby("city").agg({
    "monthly_rent": ["mean", "min", "max"],
    "record_id": "count",
})
print("Average, minimum, and maximum rent per city:")
display(summary)

Average, minimum, and maximum rent per city:


monthly_rent             record_id
                           mean   min   max     count
city                                                 
Barrie              1640.630252  1015  2490       119
Belleville          1407.378788   895  2230       132
Brantford           1544.507463   995  2385       134
Greater Sudbury     1476.984000   940  2230       125
Guelph              1640.500000  1060  2755       144
Hamilton            1522.990385   995  2460       104
Kingston            1580.258929  1080  2380       112
Kitchener-Waterloo  1706.409836   915  2720       122
London              1568.281818   985  2290       110
Oshawa              1601.857143  1050  2445       126
Ottawa              1816.859259  1185  2900       135
Peterborough        1478.094488   965  2195       127
St. Catharines      1468.616000   955  2165       125
Thunder Bay         1413.719008   895  2225       121
Toronto             1897.580882  1125  2890       136
Windsor             1349.992188   830  2015       128

In [16]:
# ---------------------------------------------------------------------
# 16. Final map — jobs, housing, and commute lines together. Combines everything above into one interactive map for the user.
# ---------------------------------------------------------------------

from housing_recommendation_module import build_settlement_map

settlement_map_osrm = build_settlement_map(
    recommendations_osrm,
    distance_provider=osrm_provider,
)
settlement_map_osrm